Aim of this script: instead of assigning a single `code_ligne` to each
*station* (see `6_Chuuchuu_data_test_line_matching.ipynb`), reconstruct the actual
physical path each *train* took across the RFN network, leg by leg, by routing
between consecutive stops over a graph built from the track geometry -- a
"chemin de moindre cout" (least-cost path) approach, using estimated travel time
(track length / speed limit) as the cost, rather than picking a single ambiguous
line per station.

Why this instead of extending notebook 6's approach further: a junction station
genuinely can't always be reduced to one `code_ligne` (notebook 6 spent five
disambiguation passes on exactly this), but the *train* did take one specific path.
Routing the whole journey sidesteps the per-station ambiguity entirely -- the
shortest path is forced to follow track that actually exists, and which of several
candidate lines the train used falls out of the routing rather than needing to be
guessed station-by-station.

This also directly answers the two follow-on questions raised when comparing this
approach to a spatial grid: "sum of trains passing through a given piece of track"
falls out of counting how many routed legs traverse each track segment, and "average
ICV/electrification for a given train" becomes a length-weighted average over the
segments its route actually took -- both computed exactly, not approximated.

Known limitations, upfront:
- This is route-matching against the RFN's *own* geometry only -- it can't route a
  leg where either endpoint station isn't itself close enough to the network (same
  `no_match`/`no_coordinates`/`international` situations as notebook 6, quantified
  below), and it can't resolve a leg whose two stations end up in different
  disconnected pieces of the graph (also quantified below).
- Where two genuinely distinct, comparably-costed routes exist between the same two
  stations, least-cost-path picks one deterministically -- it might not always be the
  one actually used. Cross-checking against repeated runs of the same named service
  (`originalRoute`) to build a consensus route, the way the map pipeline already does
  for its "modal stop sequence" trick, would sharpen this -- not implemented here, a
  natural next step.

In [ ]:
import math

import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import shapely
from shapely.ops import unary_union
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


## Step 0 -- Load the enriched Chuuchuu dataset

Same source file as notebook 6 (`data_chuuchuu_french_enriched.parquet`) -- this
notebook doesn't depend on notebook 6's `code_ligne` output at all, it's an
independent way of answering the same underlying question.

In [ ]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_enriched.parquet"

try:
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)

data_chuuchuu = data_chuuchuu.sort_values(["journey_id", "sort_time"]).reset_index(drop=True)
data_chuuchuu.shape


## Step 1 -- Station coordinates

Same station-coordinate resolution as notebook 6's Step 1 (via `sup_data/stations.csv`,
keyed on `db_id` == `deutscheBahnStopId`).

In [ ]:
unique_stations = data_chuuchuu.drop_duplicates(subset=["deutscheBahnStopId"])[
    ["deutscheBahnStopId", "stopName", "country"]
].copy()
unique_stations["db_id_str"] = unique_stations["deutscheBahnStopId"].astype(str)

data_stations = pd.read_csv("sup_data/stations.csv", sep=";", low_memory=False)
data_stations = data_stations.dropna(subset=["db_id", "latitude", "longitude"])
data_stations["db_id_str"] = data_stations["db_id"].astype("int64").astype(str)
data_stations_small = data_stations[["db_id_str", "latitude", "longitude"]].drop_duplicates(subset=["db_id_str"])

stations_with_coords = unique_stations.merge(data_stations_small, on="db_id_str", how="left")
has_coords_mask = stations_with_coords["latitude"].notna() & stations_with_coords["longitude"].notna()
stations_geo = stations_with_coords.loc[has_coords_mask].reset_index(drop=True)

stations_gdf = gpd.GeoDataFrame(
    stations_geo,
    geometry=gpd.points_from_xy(stations_geo["longitude"], stations_geo["latitude"]),
    crs="EPSG:4326",
).to_crs(2154)  # Lambert-93, metric CRS -- same choice as notebook 6

station_points = stations_gdf.set_index("deutscheBahnStopId")["geometry"]
print(f"{len(station_points)} / {len(unique_stations)} unique stations resolved to coordinates")


## Step 2 -- Load and prepare the RFN track segments

Each row of `rfn_caracteristiques.gpkg` is a "troncon" -- a piece of track with
constant attributes (`ICV`, `Vitesse`, `Electrification`, ...). Note `rg_troncon`
is **not** a reliable identifier: it restarts at 1 for every `code_ligne`, and even
within one `(code_ligne, rg_troncon)` pair multiple rows can exist (up to 15, e.g.
where `Electrification` changes partway along) -- so this uses each row's own index
as `segment_id` instead of trying to construct one from the line-numbering fields.

`Vitesse` (speed limit, km/h) has 8 nulls out of 1442 rows -- filled with the
network-wide median, since it's only used to estimate travel time for routing cost,
not reported as a real value anywhere.

In [ ]:
rfn_lines = gpd.read_file("geo_data/rfn_caracteristiques.gpkg")
rfn_lines_proj = rfn_lines.to_crs(2154)

SIMPLIFY_TOLERANCE_M = 30  # light geometry simplification for graph size / query speed --
                            # same order of magnitude as the map pipeline's ~50m simplify for
                            # a similar reason; negligible impact on computed lengths/weights

segments = rfn_lines_proj.explode(index_parts=False).reset_index(drop=True)
segments["geometry"] = segments.geometry.simplify(SIMPLIFY_TOLERANCE_M)
segments["segment_id"] = segments.index
segments["length_m"] = segments.geometry.length
segments["is_electrified"] = ~segments["Electrification"].astype(str).str.startswith("Non")
segments["Vitesse"] = segments["Vitesse"].fillna(segments["Vitesse"].median())

print(f"{len(rfn_lines_proj)} gpkg rows -> {len(segments)} single-part segments "
      f"({segments['length_m'].sum() / 1000:.0f} km total network length)")


## Step 3 -- Planarize the network

**This step is the one thing that made the difference between a graph that's
mostly disconnected and one that's actually usable.** The first version of this
notebook only snapped together segment *endpoints* that were within a tolerance of
each other -- the same style of fix as notebook 6's topology step (5e). That turned
out to be the wrong model for this data: checking the distance from every segment
endpoint to its nearest *other* endpoint showed a median of 0 m but a 75th
percentile of 135 m and a 90th percentile of ~970 m -- most endpoints don't have a
nearby matching endpoint at all. That's because a segment boundary here mostly marks
where *that segment's own* attributes change (electrification, speed limit, ...),
not where it physically joins another line -- so a great many real junctions are one
line's endpoint meeting the *middle* of another line's segment, not two endpoints
meeting each other. Endpoint-only snapping missed almost all of these: the resulting
graph had 525 fragments, and its largest connected piece held only 5% of all nodes.

The fix is proper geometric planarization: `shapely.ops.unary_union` on all segment
geometries together automatically inserts a shared node everywhere any two segments
truly touch or cross (this is standard GEOS "noding", exact geometric computation,
not a tolerance heuristic) -- splitting segments into pieces at every such point.
Explode the result into individual `LineString` pieces, then re-attach each piece to
its parent segment's attributes (`ICV`/`Vitesse`/`code_ligne`/...) by nearest-match on
each piece's midpoint against the original segments.

This still can't fix genuine *digitisation gaps* (two lines meant to connect but
sitting a real, if small, distance apart -- unary_union only nodes exact touches) --
that residual gap is handled with a tolerance-based endpoint snap in Step 4, same
idea as before, just applied to a much smaller residual problem now.

One thing this still doesn't attempt: distinguishing a genuine at-grade junction from
a grade-separated crossing (one line passing over/under another with no real
connection) -- `unary_union` nodes every geometric crossing indiscriminately. Not
checked or corrected for in this first pass.

In [ ]:
noded = unary_union(segments.geometry.values)
pieces = gpd.GeoDataFrame(geometry=list(noded.geoms), crs=2154).reset_index(drop=True)
pieces["piece_id"] = pieces.index
pieces["length_m"] = pieces.geometry.length

# re-attach each piece to its parent segment's attributes via a midpoint nearest-match
midpoints = pieces.geometry.interpolate(0.5, normalized=True)
seg_sindex = segments.sindex
attr_cols = ["segment_id", "code_ligne", "lib_ligne", "ICV", "Vitesse", "is_electrified"]

matched = []
for pt in midpoints:
    candidate_idx = list(seg_sindex.query(pt.buffer(5))) or list(seg_sindex.query(pt.buffer(50)))
    candidates = segments.iloc[candidate_idx]
    best = candidates.geometry.distance(pt).idxmin()
    matched.append(segments.loc[best, attr_cols].tolist())

attr_df = pd.DataFrame(matched, columns=attr_cols)
pieces["parent_segment_id"] = attr_df["segment_id"].values
pieces["code_ligne"] = attr_df["code_ligne"].values
pieces["lib_ligne"] = attr_df["lib_ligne"].values
pieces["ICV"] = attr_df["ICV"].values
pieces["Vitesse"] = attr_df["Vitesse"].values
pieces["is_electrified"] = attr_df["is_electrified"].values

print(f"{len(segments)} segments -> {len(pieces)} noded pieces after planarization")


## Step 4 -- Build the routable graph

Each piece is turned into a chain of graph edges between its own consecutive
vertices (not just its two endpoints) -- this keeps enough resolution along each
piece for stations to snap onto any point along it in Step 5, not just its ends.

`SNAP_TOLERANCE_M` then clusters piece endpoints that are close but not exactly
touching (the residual digitisation-gap problem Step 3 flagged) into one shared
node. This value was chosen by sweeping 100/250/500/1000/2000/5000 m and watching
the resulting graph's connectivity: 100m -> 70.6% of nodes in the largest connected
component, 250m -> 84.7%, 500m -> 91.5%, **1000m -> 94.2%**, 2000m -> 94.8%,
5000m -> 97.4%. Returns drop off sharply after 1000m while the risk of wrongly
joining genuinely separate track keeps rising with distance, so 1000m is the chosen
value -- consistent with the same "gap tolerance" role `WIDE_CONNECT_TOLERANCE_M`
(100m) played in notebook 6's topology step, just needing to be larger here since
this network was planarized from raw geometry rather than searched around an
already-known anchor line.

Edge weight for routing is **estimated travel time** (`length_m / (Vitesse_km/h)`),
not raw distance -- a TGV taking a longer LGV detour because it's faster is exactly
the kind of real routing behaviour that a pure-distance cost would get wrong.

In [ ]:
SNAP_TOLERANCE_M = 1000  # residual endpoint-gap tolerance, see markdown above for how this was chosen

edge_rows = []       # (u, v, length_m, piece_id)
vertex_rows = []     # (node_id, x, y) -- every vertex of every piece, needed for station snapping later
endpoint_rows = []   # (piece_id, node_id, x, y) -- just first/last vertex of each piece, for junction snapping
next_node_id = 0

for pc in pieces.itertuples():
    coords = list(pc.geometry.coords)
    node_ids = list(range(next_node_id, next_node_id + len(coords)))
    next_node_id += len(coords)

    for node_id, (x, y) in zip(node_ids, coords):
        vertex_rows.append((node_id, x, y))
    for i in range(len(coords) - 1):
        length = math.dist(coords[i], coords[i + 1])
        edge_rows.append((node_ids[i], node_ids[i + 1], length, pc.piece_id))

    endpoint_rows.append((pc.piece_id, node_ids[0], coords[0][0], coords[0][1]))
    endpoint_rows.append((pc.piece_id, node_ids[-1], coords[-1][0], coords[-1][1]))

edges_raw = pd.DataFrame(edge_rows, columns=["u", "v", "length_m", "piece_id"])
vertices = pd.DataFrame(vertex_rows, columns=["node_id", "x", "y"])
endpoints = pd.DataFrame(endpoint_rows, columns=["piece_id", "node_id", "x", "y"])
print(f"{next_node_id} raw vertex nodes, {len(edges_raw)} raw edges, {len(endpoints)} piece-endpoints to snap")


In [ ]:
# cluster piece-endpoints within SNAP_TOLERANCE_M into one shared canonical node each
tree = cKDTree(endpoints[["x", "y"]].to_numpy())
close_pairs = tree.query_pairs(r=SNAP_TOLERANCE_M, output_type="ndarray")

adjacency = coo_matrix(
    (np.ones(len(close_pairs)), (close_pairs[:, 0], close_pairs[:, 1])),
    shape=(len(endpoints), len(endpoints)),
)
n_clusters, cluster_labels = connected_components(adjacency, directed=False)
endpoints["cluster_id"] = cluster_labels

CLUSTER_NODE_OFFSET = next_node_id  # keeps clustered node ids clear of raw vertex node ids
endpoints["canonical_node"] = endpoints["cluster_id"] + CLUSTER_NODE_OFFSET
node_remap = dict(zip(endpoints["node_id"], endpoints["canonical_node"]))

edges = edges_raw.copy()
edges["u"] = edges["u"].map(node_remap).fillna(edges["u"]).astype(int)
edges["v"] = edges["v"].map(node_remap).fillna(edges["v"]).astype(int)
edges = edges[(edges["u"] != edges["v"]) & (edges["length_m"] > 0)].reset_index(drop=True)

vertices["canonical_node"] = vertices["node_id"].map(node_remap).fillna(vertices["node_id"]).astype(int)
node_coords = vertices.groupby("canonical_node")[["x", "y"]].mean()  # centroid for clustered nodes, exact point otherwise

print(f"{len(endpoints)} piece-endpoints -> {n_clusters} distinct junction/end nodes within {SNAP_TOLERANCE_M} m")


In [ ]:
attr_cols = ["piece_id", "parent_segment_id", "code_ligne", "lib_ligne", "ICV", "Vitesse", "is_electrified"]
edges = edges.merge(pieces[attr_cols], on="piece_id", how="left")
edges["time_s"] = edges["length_m"] / (edges["Vitesse"] * 1000 / 3600)  # km/h -> m/s

# a real column, not a reliance on positional-index alignment surviving later merges --
# this is the identifier used both as the MultiGraph edge key and (in Step 5) as
# edges_gdf's index, so a specific edge can always be referenced unambiguously
edges["edge_id"] = edges.index

network_graph = nx.MultiGraph()
for row in edges.itertuples():
    network_graph.add_edge(
        row.u, row.v, key=row.edge_id,
        length_m=row.length_m, time_s=row.time_s,
        piece_id=row.piece_id, parent_segment_id=row.parent_segment_id,
        code_ligne=row.code_ligne, ICV=row.ICV, is_electrified=row.is_electrified,
    )

n_components = nx.number_connected_components(network_graph)
largest_component_size = len(max(nx.connected_components(network_graph), key=len))
print(f"graph: {network_graph.number_of_nodes()} nodes, {network_graph.number_of_edges()} edges")
print(f"{n_components} connected components; largest has {largest_component_size} nodes "
      f"({largest_component_size / network_graph.number_of_nodes():.1%} of all nodes)")


## Step 5 -- Snap stations onto the graph

For each station, find the nearest graph *edge* (not just nearest node) within
`STATION_SNAP_MAX_M`, and insert a new node at the projected point, splitting that
edge in two. This is deliberately the same kind of operation as Step 4's junction
snapping, just applied to stations instead of track endpoints -- a station on a
line between two junctions needs to attach to the middle of an edge, not to either
end of it.

If two stations happen to snap onto the *same* edge (plausible for near-duplicate
stops close together, e.g. "Lutterbach" / "Lutterbach Tram-train" from notebook 6's
ambiguous-station list), that edge is split into a chain through all of them in
along-edge order, not just split once -- handled per group below rather than
one-at-a-time, since splitting invalidates the original edge for anyone processed
after it.

In [ ]:
STATION_SNAP_MAX_M = 300  # station must be within this distance of the network to attach to it

edges_with_coords = (
    edges.merge(node_coords.add_suffix("_u"), left_on="u", right_index=True)
    .merge(node_coords.add_suffix("_v"), left_on="v", right_index=True)
)
coords_stack = np.stack(
    [edges_with_coords[["x_u", "y_u"]].to_numpy(), edges_with_coords[["x_v", "y_v"]].to_numpy()], axis=1
)
edge_geoms = shapely.linestrings(coords_stack)
edges_gdf = gpd.GeoDataFrame(edges_with_coords, geometry=edge_geoms, crs=2154).set_index("edge_id")

edge_sindex = edges_gdf.sindex
snap_rows = []
for stop_id, point in station_points.items():
    candidate_idx = list(edge_sindex.query(point.buffer(STATION_SNAP_MAX_M)))
    if not candidate_idx:
        continue
    candidates = edges_gdf.iloc[candidate_idx]
    dist_to_edge = candidates.geometry.distance(point)
    best_idx = dist_to_edge.idxmin()
    if dist_to_edge.loc[best_idx] > STATION_SNAP_MAX_M:
        continue
    best_edge = edges_gdf.loc[best_idx]
    dist_from_u = best_edge.geometry.project(point)
    snap_rows.append({
        "deutscheBahnStopId": stop_id, "edge_key": best_idx,
        "u": int(best_edge["u"]), "v": int(best_edge["v"]),
        "dist_from_u": dist_from_u, "dist_to_network_m": dist_to_edge.loc[best_idx],
    })

station_snap = pd.DataFrame(snap_rows)
print(f"{len(station_snap)} / {len(station_points)} stations snapped onto the network within {STATION_SNAP_MAX_M} m")


In [ ]:
station_node = {}
next_node_id_counter = int(node_coords.index.max()) + 1

for edge_key, group in station_snap.groupby("edge_key"):
    u, v = int(group["u"].iloc[0]), int(group["v"].iloc[0])
    edge_attrs = dict(network_graph.get_edge_data(u, v, key=edge_key))
    total_length, total_time = edge_attrs["length_m"], edge_attrs["time_s"]
    shared_attrs = {k: val for k, val in edge_attrs.items() if k not in ("length_m", "time_s")}

    ordered = group.sort_values("dist_from_u")
    new_nodes = []
    for _, rec in ordered.iterrows():
        node_id = next_node_id_counter
        next_node_id_counter += 1
        proj_point = edges_gdf.loc[edge_key, "geometry"].interpolate(rec["dist_from_u"])
        node_coords.loc[node_id] = [proj_point.x, proj_point.y]
        station_node[rec["deutscheBahnStopId"]] = node_id
        new_nodes.append((node_id, rec["dist_from_u"]))

    network_graph.remove_edge(u, v, key=edge_key)
    chain = [(u, 0.0)] + new_nodes + [(v, total_length)]
    for (node_a, dist_a), (node_b, dist_b) in zip(chain, chain[1:]):
        # never skip a sub-edge even if two stations project to (almost) the same point --
        # dropping it would leave a station node with no edge at all, disconnected from the
        # graph entirely rather than just imprecisely located. A zero-length edge is harmless
        # for shortest-path purposes (it just contributes no extra cost).
        seg_length = max(dist_b - dist_a, 0.0)
        frac = seg_length / total_length if total_length > 0 else 0.0
        network_graph.add_edge(node_a, node_b, length_m=seg_length, time_s=total_time * frac, **shared_attrs)

print(f"{len(station_node)} station nodes inserted into the graph")


## Step 6 -- Identify unique consecutive-stop legs

A "leg" is one train's hop from one stop to the very next stop on the same
journey. The same pair of stations recurs constantly across different dates/trains
(e.g. the daily Lyon-Toulouse TGV), so routing is only computed once per **unique**
unordered `(stop_a, stop_b)` pair, then broadcast back -- across the full dataset
this is only ~8,650 unique pairs behind ~6.18M leg-rows, the same
compute-once-per-unique-combination pattern used throughout notebook 6.

In [ ]:
next_stop_id = data_chuuchuu.groupby("journey_id")["deutscheBahnStopId"].shift(-1)
has_next = next_stop_id.notna()

leg_rows = pd.DataFrame({
    "stop_a": data_chuuchuu.loc[has_next, "deutscheBahnStopId"],
    "stop_b": next_stop_id[has_next],
})
leg_rows["stop_lo"] = np.minimum(leg_rows["stop_a"], leg_rows["stop_b"])
leg_rows["stop_hi"] = np.maximum(leg_rows["stop_a"], leg_rows["stop_b"])

unique_legs = leg_rows[["stop_lo", "stop_hi"]].drop_duplicates().reset_index(drop=True)
print(f"{len(leg_rows)} leg-rows across the full dataset -> {len(unique_legs)} unique station pairs to route")


## Step 7 -- Route each unique leg (least-cost path)

For each unique `(stop_lo, stop_hi)` pair: shortest path by estimated travel time
(`weight="time_s"`) between their snapped graph nodes. `networkx` handles the
`MultiGraph`'s parallel edges (e.g. classic line vs. LGV both connecting the same
two junctions) transparently for a string `weight` -- it takes the minimum-weight
edge among parallels automatically, both when computing the path and in the
`min(..., key=...)` used below to re-extract which specific edge that was.

From the path, `avg_ICV` and `pct_electrified` are length-weighted over the whole
route -- summing each traversed edge's `length_m x ICV` (or `x is_electrified`) and
dividing by total route length -- not a plain per-edge average, since a 500 m siding
and a 50 km main line shouldn't count equally.

`routing_status` distinguishes *why* a leg didn't route: `missing_station_node`
(one or both stations never snapped onto the network -- the same
`no_match`/`no_coordinates`/`international` situations as notebook 6) versus
`no_path` (both stations are on the network, but in different connected pieces of
it -- a real residual gap, not a missing-data issue).

In [ ]:
def route_leg(stop_lo, stop_hi):
    u = station_node.get(stop_lo)
    v = station_node.get(stop_hi)
    if u is None or v is None:
        return {"routing_status": "missing_station_node"}
    try:
        path_nodes = nx.shortest_path(network_graph, u, v, weight="time_s")
    except nx.NetworkXNoPath:
        return {"routing_status": "no_path"}
    except nx.NodeNotFound:
        # defensive: a station node that got inserted into station_node but somehow never
        # connected to the graph -- shouldn't happen given how Step 5 builds it, but fail
        # visibly into the status breakdown rather than crashing the whole routing loop
        return {"routing_status": "node_not_in_graph"}

    total_length = 0.0
    weighted_icv_sum = 0.0
    electrified_length = 0.0
    segment_ids = []
    for a, b in zip(path_nodes, path_nodes[1:]):
        edge_data = min(network_graph.get_edge_data(a, b).values(), key=lambda d: d["time_s"])
        total_length += edge_data["length_m"]
        weighted_icv_sum += edge_data["length_m"] * edge_data["ICV"]
        if edge_data["is_electrified"]:
            electrified_length += edge_data["length_m"]
        segment_ids.append(edge_data["parent_segment_id"])

    if total_length == 0:
        return {"routing_status": "zero_length_path"}

    return {
        "routing_status": "routed",
        "route_length_m": total_length,
        "avg_ICV": weighted_icv_sum / total_length,
        "pct_electrified": electrified_length / total_length,
        "n_segments": len(dict.fromkeys(segment_ids)),
        "segment_ids": list(dict.fromkeys(segment_ids)),  # unique, order-preserving
        "path_nodes": path_nodes,  # ordered node ids -- reconstructed into lon/lat in the next step
    }


leg_results = [route_leg(lo, hi) for lo, hi in unique_legs.itertuples(index=False)]

leg_routing = unique_legs.copy()
for col in ["routing_status", "route_length_m", "avg_ICV", "pct_electrified", "n_segments", "segment_ids", "path_nodes"]:
    leg_routing[col] = [r.get(col) for r in leg_results]

leg_routing["routing_status"].value_counts(dropna=False)


In [ ]:
# Reconstruct each routed leg's actual path geometry, for map visualization / visual
# verification -- a leg's shortest path can travel through only a short stretch of a
# long code_ligne, so highlighting the whole line (the map's original approach)
# hugely overstates a train's real footprint. Comparing route_length_m against the
# straight-line distance between the two stations confirmed this matters: most legs
# have a route/straight ratio near 1 (a direct path), but a small, real subset (~2%
# of leg-ROWS) route hundreds of km for stations only a few km apart -- a network
# graph artifact (likely a missing connector / bad snap) worth surfacing directly on
# the map rather than masking it behind a whole-line highlight.
node_points_wgs84 = gpd.GeoSeries(
    gpd.points_from_xy(node_coords["x"], node_coords["y"]), index=node_coords.index, crs=2154
).to_crs(4326)
node_lonlat = {node_id: (round(pt.x, 5), round(pt.y, 5)) for node_id, pt in node_points_wgs84.items()}

def path_to_coords(nodes):
    if not isinstance(nodes, list):
        return None
    return [list(node_lonlat[n]) for n in nodes]

leg_routing["path_coords"] = leg_routing["path_nodes"].apply(path_to_coords)
print(f"{leg_routing['path_coords'].notna().sum()} / {len(leg_routing)} routed legs have path geometry")


### Verification -- routing coverage vs. the number of leg-rows it actually explains

`unique_legs` is a count of distinct *pairs*; weighting by how many leg-rows each
pair actually accounts for (`leg_rows`, not `unique_legs`) shows what fraction of
the full dataset's train-hops this can actually attribute an infrastructure profile
to.

In [ ]:
leg_occurrence_counts = leg_rows.groupby(["stop_lo", "stop_hi"]).size().rename("n_occurrences").reset_index()
leg_routing = leg_routing.merge(leg_occurrence_counts, on=["stop_lo", "stop_hi"], how="left")

coverage = leg_routing.groupby("routing_status")["n_occurrences"].sum().sort_values(ascending=False)
print("leg-rows by routing outcome:")
print(coverage)
print(f"\n{coverage.get('routed', 0) / coverage.sum():.1%} of all leg-rows in the dataset were routed")


## Step 8 -- Broadcast leg-level results back onto the full dataset

Only the scalar summary columns are broadcast onto the full ~7M-row table (not
`segment_ids`, which is kept in the small `leg_routing` lookup table only) --
broadcasting a list column onto every one of 7M rows would bloat the export for
little benefit; anyone needing the exact segment path for a leg can join back to
`leg_routing` on `(stop_lo, stop_hi)`.

In [ ]:
# reuse Step 6's next_stop_id/has_next rather than recomputing -- terminus rows have no
# next stop (next_stop_id is null there), so stop_lo/stop_hi must only be computed on the
# has_next subset, otherwise np.minimum chokes comparing a string against NaN
data_chuuchuu["stop_lo"] = pd.Series(pd.NA, index=data_chuuchuu.index, dtype="object")
data_chuuchuu["stop_hi"] = pd.Series(pd.NA, index=data_chuuchuu.index, dtype="object")
data_chuuchuu.loc[has_next, "stop_lo"] = np.minimum(
    data_chuuchuu.loc[has_next, "deutscheBahnStopId"], next_stop_id[has_next]
)
data_chuuchuu.loc[has_next, "stop_hi"] = np.maximum(
    data_chuuchuu.loc[has_next, "deutscheBahnStopId"], next_stop_id[has_next]
)

broadcast_cols = ["stop_lo", "stop_hi", "routing_status", "route_length_m", "avg_ICV", "pct_electrified", "n_segments"]
data_chuuchuu = data_chuuchuu.merge(leg_routing[broadcast_cols], on=["stop_lo", "stop_hi"], how="left")

data_chuuchuu["routing_status"] = data_chuuchuu["routing_status"].fillna("terminus_no_next_stop")
data_chuuchuu["routing_status"].value_counts(dropna=False)


## Step 9 -- Roll up to journey level

Combining leg-level `avg_ICV` back up to a whole-journey average re-weights by each
leg's own `route_length_m` -- since each leg's `avg_ICV` is already itself
length-weighted internally, this preserves the true grand length-weighted average
over the whole trip (not a naive mean-of-means that would over-count short legs).

Deliberately kept as a leg-level-first computation, not the only level available:
a single whole-journey average can dilute a short stretch of degraded track into
invisibility next to hundreds of km of good track. Anyone wanting to correlate
infrastructure quality against delay is likely better served working at the leg
level (which already has arrival/departure delay per stop in the source data) than
starting from this journey-level rollup.

In [ ]:
def weighted_avg(values, weights):
    weights = weights.where(values.notna())
    return (values * weights).sum() / weights.sum() if weights.sum() > 0 else np.nan


journey_rollup = data_chuuchuu.groupby("journey_id").apply(
    lambda g: pd.Series({
        "n_legs_total": g["routing_status"].ne("terminus_no_next_stop").sum(),
        "n_legs_routed": (g["routing_status"] == "routed").sum(),
        "total_route_length_m": g["route_length_m"].sum(),
        "avg_ICV": weighted_avg(g["avg_ICV"], g["route_length_m"]),
        "pct_electrified": weighted_avg(g["pct_electrified"], g["route_length_m"]),
    }),
    include_groups=False,
).reset_index()

journey_rollup["pct_legs_routed"] = journey_rollup["n_legs_routed"] / journey_rollup["n_legs_total"]
print(f"{len(journey_rollup)} journeys")
journey_rollup["pct_legs_routed"].describe()


## Step 10 -- Per-segment traffic counts

For every routed leg, its occurrence count (`n_occurrences` from Step 7's
verification, i.e. how many actual train-stop-events used that leg) is added to
every segment its route passed through -- giving a count of how many recorded
train-hops physically traversed each piece of track, directly from the routed
paths rather than approximated via any kind of spatial binning.

In [ ]:
traffic_rows = leg_routing.loc[leg_routing["routing_status"] == "routed", ["segment_ids", "n_occurrences"]]
exploded_traffic = traffic_rows.explode("segment_ids").rename(columns={"segment_ids": "segment_id"})
segment_traffic = exploded_traffic.groupby("segment_id")["n_occurrences"].sum().rename("n_train_passages")

segments_with_traffic = segments.merge(segment_traffic, left_on="segment_id", right_index=True, how="left")
segments_with_traffic["n_train_passages"] = segments_with_traffic["n_train_passages"].fillna(0).astype(int)

print(f"{(segments_with_traffic['n_train_passages'] > 0).sum()} / {len(segments_with_traffic)} "
      f"segments have at least one recorded train passage")
segments_with_traffic.sort_values("n_train_passages", ascending=False)[
    ["code_ligne", "lib_ligne", "n_train_passages", "ICV", "is_electrified"]
].head(15)


### Verification -- regression check against the Valence TGV case from notebook 6

Notebook 6's direction-of-travel step originally mis-resolved Valence TGV (on
journey `TGV INOUI 6823`, Lyon Part-Dieu -> Toulouse Matabiau) to line `908000`
("Ligne de Valence a Moirans", heading back north-east toward Grenoble) instead of
the correct `752000` (the LGV continuing south) -- caught by manual inspection and
fixed by switching to a nearest-real-neighbour reference axis. Since this notebook's
routing is a completely independent method, re-checking that specific leg
(Lyon Part-Dieu -> Valence TGV) here is a good cross-check that both approaches
agree on a case we already know the right answer to.

In [ ]:
lyon_part_dieu = "8700152"
valence_tgv = "8704943"

check_lo, check_hi = sorted([lyon_part_dieu, valence_tgv])
check_row = leg_routing.loc[(leg_routing["stop_lo"] == check_lo) & (leg_routing["stop_hi"] == check_hi)]
print(check_row[["routing_status", "route_length_m", "avg_ICV", "pct_electrified", "n_segments"]].to_string())

if len(check_row) and check_row["routing_status"].iloc[0] == "routed":
    used_lines = segments.loc[
        segments["segment_id"].isin(check_row["segment_ids"].iloc[0]), "code_ligne"
    ].unique()
    print(f"\nlines used on the routed Lyon Part-Dieu <-> Valence TGV leg: {sorted(used_lines)}")
    print("expect 752000 to appear; 908000 should NOT appear")


## Exporting data

In [ ]:
export_data = input("Export network-routing outputs to parquet? (y/n): ")

if export_data.lower() == "y":
    import os
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    leg_routing.to_parquet(f"{intermediate_outputs_dir}/network_routing_legs.parquet")

    journey_rollup.to_parquet(f"{intermediate_outputs_dir}/network_routing_journeys.parquet")

    segments_with_traffic.drop(columns="geometry").to_parquet(
        f"{intermediate_outputs_dir}/network_routing_segment_traffic.parquet"
    )
    segments_with_traffic[["segment_id", "geometry"]].to_file(
        f"{intermediate_outputs_dir}/network_routing_segment_traffic.gpkg", driver="GPKG"
    )

    print("exported: network_routing_legs.parquet, network_routing_journeys.parquet, "
          "network_routing_segment_traffic.parquet (+ .gpkg for mapping)")
